# Snippet from Verification.md


In [ ]:
# Adapted from tests/invariants/test_invariants_energy.py (real file, mocked deps)
import pytest
import numpy as np
from unittest.mock import MagicMock
from compitum.energy import SymbolicFreeEnergy
from compitum.predictors import CalibratedPredictor
from hypothesis import given, strategies as st

@pytest.mark.invariants
def test_utility_improves_with_quality():
    energy_func = SymbolicFreeEnergy(alpha=0.4, beta_t=0.2, beta_c=0.15, beta_d=0.15, beta_s=0.1)
    mock_metric = MagicMock()
    mock_metric.distance.return_value = (1.0, 0.1)  # (d, sigma)
    mock_coherence = MagicMock()
    mock_coherence.log_evidence.return_value = 0.0
    mock_model = MagicMock()
    mock_model.cost = 0.0
    mock_predictors = {
        "quality": MagicMock(spec=CalibratedPredictor),
        "latency": MagicMock(spec=CalibratedPredictor),
        "cost": MagicMock(spec=CalibratedPredictor),
    }
    for p in mock_predictors.values():
        p.predict.return_value = (np.array([0.5]), np.array([0.4]), np.array([0.6]))

    U_t, _, _ = energy_func.compute(np.zeros(1), mock_model, mock_predictors, mock_coherence, mock_metric)

    # Improve the quality predictor's forecast; utility should not decrease
    mock_predictors["quality"].predict.return_value = (np.array([0.9]), np.array([0.8]), np.array([1.0]))
    U_tp1, _, _ = energy_func.compute(np.zeros(1), mock_model, mock_predictors, mock_coherence, mock_metric)

    assert U_tp1 >= U_t, "Utility should not decrease when quality improves"

@given(st.lists(st.floats(min_value=0, max_value=1), min_size=2))
def test_feas_slack_nonneg(utilities):
    # Prop: Sorted utils -> gap >=0
    sorted_u = np.sort(utilities)[::-1]
    assert sorted_u[0] >= sorted_u[1], "Gap negative-infeas order"
